# Kimi K2 Debug Test

Testing Kimi K2 exactly like the working test notebook to debug agent issues.


In [1]:
%pip install requests python-dotenv openai

import json
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


True

In [2]:
# Initialize Kimi client EXACTLY like test notebook
api_key = os.getenv("MOONSHOT_API_KEY")
if not api_key:
    raise ValueError("MOONSHOT_API_KEY not found")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.moonshot.ai/v1"
)

model = "kimi-k2-thinking"
print(f"✅ Client initialized")
print(f"Model: {model}")


✅ Client initialized
Model: kimi-k2-thinking


In [3]:
# Test 1: Simple query (like test notebook)
query = "我需要讨论潜规则问题，这个在中国商业环境中很重要。Can you explain what this means in English and provide cultural context?"

system_prompt = """You are a bilingual AI assistant specialized in Chinese-English code-switching discourse.

Task: Preserve culturally-loaded Chinese terms (潜规则, 关系, 面子) in English translation.

Theory:
- Myers-Scotton (1993) - Matrix Language Frame Model
- Sperber & Wilson (1986) - Relevance Theory

Input: Mixed-language query
Output: JSON with {"english_response": "...", "cultural_notes": "..."}

Preserve cultural context and explain untranslatable concepts."""

# Build full query with system prompt inline
full_query = f"{system_prompt}\n\nQuery: {query}"

print(f"Query length: {len(full_query)} chars")
print(f"\nQuery preview: {full_query[:200]}...")


Query length: 552 chars

Query preview: You are a bilingual AI assistant specialized in Chinese-English code-switching discourse.

Task: Preserve culturally-loaded Chinese terms (潜规则, 关系, 面子) in English translation.

Theory:
- Myers-Scotton...


In [4]:
# Test 1: Simple messages array (NO system role) - EXACTLY like test notebook
messages = [{"role": "user", "content": full_query}]

print("Testing with simple user message (no system role)...")
print(f"Messages: {messages}")
print("\n" + "="*70)

try:
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.7,
    )
    
    print(f"✅ Response received")
    print(f"Response type: {type(response)}")
    print(f"Has choices: {hasattr(response, 'choices')}")
    
    if response.choices:
        choice = response.choices[0]
        print(f"Finish reason: {choice.finish_reason}")
        
        message = choice.message
        print(f"Message type: {type(message)}")
        print(f"Message attributes: {dir(message)[:15]}")
        
        content = message.content
        print(f"\nContent type: {type(content)}")
        print(f"Content value: {repr(content)}")
        print(f"Content length: {len(content) if content else 0}")
        
        if content:
            print(f"\n✅ Content preview: {content[:200]}...")
        else:
            print(f"\n❌ Content is empty or None!")
            print(f"Full message object: {message}")
            
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()


Testing with simple user message (no system role)...
Messages: [{'role': 'user', 'content': 'You are a bilingual AI assistant specialized in Chinese-English code-switching discourse.\n\nTask: Preserve culturally-loaded Chinese terms (潜规则, 关系, 面子) in English translation.\n\nTheory:\n- Myers-Scotton (1993) - Matrix Language Frame Model\n- Sperber & Wilson (1986) - Relevance Theory\n\nInput: Mixed-language query\nOutput: JSON with {"english_response": "...", "cultural_notes": "..."}\n\nPreserve cultural context and explain untranslatable concepts.\n\nQuery: 我需要讨论潜规则问题，这个在中国商业环境中很重要。Can you explain what this means in English and provide cultural context?'}]

✅ Response received
Response type: <class 'openai.types.chat.chat_completion.ChatCompletion'>
Has choices: True
Finish reason: stop
Message type: <class 'openai.types.chat.chat_completion_message.ChatCompletionMessage'>
Message attributes: ['__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__class_vars__', '_

In [5]:
# Test 2: Check if there's a different way to access content when finish_reason is "length"
test_query = "Say hello in Chinese and English."
messages_test = [{"role": "user", "content": test_query}]

print("Testing minimal query...")
print("\n" + "="*70)

try:
    response = client.chat.completions.create(
        model=model,
        messages=messages_test,
        temperature=0.7,
        max_tokens=100
    )
    
    print(f"\nFull response object:")
    print(f"Type: {type(response)}")
    print(f"Dir: {[x for x in dir(response) if not x.startswith('_')]}")
    
    if response.choices:
        choice = response.choices[0]
        print(f"\nChoice finish_reason: {choice.finish_reason}")
        
        message = choice.message
        print(f"\nMessage type: {type(message)}")
        print(f"Message dir: {[x for x in dir(message) if not x.startswith('_')]}")
        
        # Try all possible ways to get content
        if hasattr(message, 'content'):
            print(f"message.content: {repr(message.content)}")
        if hasattr(message, 'text'):
            print(f"message.text: {repr(message.text)}")
        if hasattr(message, 'message'):
            print(f"message.message: {repr(message.message)}")
        
        # Print full message dict if possible
        try:
            msg_dict = message.model_dump() if hasattr(message, 'model_dump') else str(message)
            print(f"\nMessage dict/model_dump: {msg_dict}")
        except:
            print(f"\nMessage str: {str(message)}")
            
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()


Testing minimal query...


Full response object:
Type: <class 'openai.types.chat.chat_completion.ChatCompletion'>
Dir: ['choices', 'construct', 'copy', 'created', 'dict', 'from_orm', 'id', 'json', 'model', 'model_computed_fields', 'model_config', 'model_construct', 'model_copy', 'model_dump', 'model_dump_json', 'model_extra', 'model_fields', 'model_fields_set', 'model_json_schema', 'model_parametrized_name', 'model_post_init', 'model_rebuild', 'model_validate', 'model_validate_json', 'model_validate_strings', 'object', 'parse_file', 'parse_obj', 'parse_raw', 'schema', 'schema_json', 'service_tier', 'system_fingerprint', 'to_dict', 'to_json', 'update_forward_refs', 'usage', 'validate']

Choice finish_reason: length

Message type: <class 'openai.types.chat.chat_completion_message.ChatCompletionMessage'>
Message dir: ['annotations', 'audio', 'construct', 'content', 'copy', 'dict', 'from_orm', 'function_call', 'json', 'model_computed_fields', 'model_config', 'model_construct', 'model_copy'